# Feature Engineering

This notebook converts the raw Steam dataset into the modeling table used by the training and evaluation notebooks.


In [1]:
import ast
import csv
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 140)

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "games.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"


## 1. Load and clean raw data

The raw CSV has a malformed header where `Discount` and `DLC count` are merged. The loader repairs that first, then applies basic type conversions and normalizes blank strings.

In [3]:
RAW_DATA_PATH = DATA_DIR / "games.csv"
OUTPUT_PATH = DATA_DIR / "games_price_model.csv"

NUMERIC_COLS = [
    "Peak CCU",
    "Required age",
    "Price",
    "Discount",
    "DLC count",
    "Metacritic score",
    "User score",
    "Positive",
    "Negative",
    "Score rank",
    "Achievements",
    "Recommendations",
    "Average playtime forever",
    "Average playtime two weeks",
    "Median playtime forever",
    "Median playtime two weeks",
]

BOOL_COLS = ["Windows", "Mac", "Linux"]
TEXT_COLS = [
    "Name",
    "Estimated owners",
    "About the game",
    "Supported languages",
    "Full audio languages",
    "Reviews",
    "Website",
    "Support url",
    "Support email",
    "Metacritic url",
    "Developers",
    "Publishers",
    "Categories",
    "Genres",
    "Tags",
    "Screenshots",
    "Movies",
    "Notes",
]


def load_games_csv(path):
    rows = []
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        raw_header = next(reader)
        header = raw_header[:7] + ["Discount", "DLC count"] + raw_header[8:]

        for raw_row in reader:
            if len(raw_row) != len(header):
                raise ValueError(f"Unexpected row length {len(raw_row)}")
            rows.append(raw_row)

    df = pd.DataFrame(rows, columns=header)

    for col in TEXT_COLS:
        if col in df.columns:
            df[col] = df[col].replace("", np.nan)

    df["AppID"] = pd.to_numeric(df["AppID"], errors="coerce").astype("Int64")
    for col in NUMERIC_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in BOOL_COLS:
        df[col] = df[col].map({"True": True, "False": False})

    df["Release date parsed"] = pd.to_datetime(df["Release date"], format="%b %d, %Y", errors="coerce")
    df["Release year"] = df["Release date parsed"].dt.year

    return df


df_raw = load_games_csv(RAW_DATA_PATH)
print(f"Raw shape: {df_raw.shape}")
df_raw.head()

Raw shape: (122611, 42)


,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,Discount,DLC count,About the game,Supported languages,Full audio languages,Reviews,Header image,Website,Support url,Support email,Windows,Mac,Linux,Metacritic score,Metacritic url,User score,Positive,Negative,Score rank,Achievements,Recommendations,Notes,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies,Release date parsed,Release year
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,[],[],NaN,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/2539430/header.jpg?t=1699268702,NaN,NaN,NaN,True,False,False,0,NaN,0,0,0,NaN,0,0,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/2539430/ss_7d9504b958d0b143d053d31cb74b375daba338d6.1920x1080.jpg?t=1...,NaN,2023-08-01,2023
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come into full bloom. The protagonist Yukinari Sanada has returned to his hometown in Kanagawa ...",['English'],[],NaN,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/496350/header.jpg?t=1725519097,http://mangagamer.org/supipara,http://mangagamer.com,support@mangagamer.com,True,False,False,0,NaN,0,252,3,NaN,0,231,NaN,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,Family Sharing",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/496350/ss_0a5ff85870a1c2221e495b75a5c7b9e248eb3249.1920x1080.jpg?t=17...,NaN,2016-07-29,2016
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical and entrancing world of Edgar Allan Poe's ''The Raven!'' This work reflected the image of...","['English', 'French', 'German', 'Russian']",[],NaN,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1034400/header.jpg?t=1687434115,https://www.facebook.com/8FloorGames/,https://www.facebook.com/8FloorGames,support@8floor.net,True,True,False,0,NaN,0,21,3,NaN,0,0,NaN,0,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Object,2D,Colorful,Stylized,Logic,Mystery,Atmospheric,Family Friendly,PvE,Tutorial,Singleplayer...",https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/1034400/ss_12d8bb27a16046fe548818dcd0f5d7d0e2038179.1920x1080.jpg?t=1...,NaN,2019-05-06,2019
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' It's been three years since I've repeated such greetings. Hiyoro, who couldn't avoid Hakko...",['Korean'],['Korean'],NaN,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/3292190/header.jpg?t=1742217574,NaN,NaN,yujingamesc@gmail.com,True,False,False,0,NaN,0,0,0,NaN,19,0,The game includes the following elements. 1. General Mature Content : Revealing outfits; sexual stimulation; sexual innuendo; sex-relate...,0,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/3292190/ss_87f071dbf846e7832ff49d68b0d7a8b6509bcda6.1920x1080.jpg?t=1...,NaN,2024-10-31,2024
4,3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,"Its not just a Maze; its a Quest! Enter the captivating realm of Maze Quest, where each unexpected turn presents fresh challenges, hidde...",['English'],['English'],NaN,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/3631080/2a78ae30e350862e415889f097e422855eca25b9/header.jpg?t=1746637727,https://www.realityexpanded.com/books-games,https://www.realityexpanded.com,support@realityexpanded.com,True,False,False,0,NaN,0,0,0,NaN,0,0,NaN,0,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderbo

## 2. Build price target

`Price` is the current Steam price, which may already include an active discount. For market-aligned pricing, the main target should be estimated list price.

In [4]:
df = df_raw.copy()

df["current_price"] = df["Price"]
df["has_active_discount"] = df["Discount"].gt(0)

conditions = [
    df["Price"].gt(0) & df["Discount"].eq(0),
    df["Price"].gt(0) & df["Discount"].gt(0) & df["Discount"].lt(100),
]
values = [
    df["Price"],
    df["Price"] / (1 - df["Discount"] / 100),
]

df["estimated_list_price"] = np.select(conditions, values, default=np.nan)
df["estimated_list_price"] = pd.Series(df["estimated_list_price"], index=df.index).round(2)

df["price_target_source"] = np.select(
    [
        df["Price"].gt(0) & df["Discount"].eq(0),
        df["Price"].gt(0) & df["Discount"].gt(0) & df["Discount"].lt(100),
        df["Price"].eq(0),
        df["Discount"].eq(100),
    ],
    ["observed", "reconstructed", "free", "invalid_discount_100"],
    default="invalid",
)

df["valid_price_target"] = df["estimated_list_price"].between(0.49, 100, inclusive="both")
df["log_list_price"] = np.where(df["valid_price_target"], np.log1p(df["estimated_list_price"]), np.nan)

target_summary = (
    df.groupby("price_target_source")
    .agg(
        games=("AppID", "size"),
        valid_targets=("valid_price_target", "sum"),
        median_current_price=("current_price", "median"),
        median_estimated_list_price=("estimated_list_price", "median"),
    )
    .sort_values("games", ascending=False)
)
target_summary

,games,valid_targets,median_current_price,median_estimated_list_price
price_target_source,,,,
observed,55763,55464,3.99,3.99
reconstructed,40640,40598,2.79,7.97
free,26206,0,0.00,NaN
invalid_discount_100,2,0,0.64,NaN


In [5]:
print("Valid paid price targets:", df["valid_price_target"].sum())
print("Invalid Discount == 100 rows:", df["price_target_source"].eq("invalid_discount_100").sum())
print("Estimated list price > 100 rows:", df["estimated_list_price"].gt(100).sum())

df.loc[df["estimated_list_price"].gt(100), [
    "AppID", "Name", "current_price", "Discount", "estimated_list_price", "Genres", "Recommendations"
]].sort_values("estimated_list_price", ascending=False).head(20)

Valid paid price targets: 96062
Invalid Discount == 100 rows: 2
Estimated list price > 100 rows: 341


,AppID,Name,current_price,Discount,estimated_list_price,Genres,Recommendations
69662,3230900,Hidden Storehouse Top-Down 3D,199.99,95,3999.80,"Action,Adventure,Casual,Indie,Strategy",0
55868,1834290,Toy War - Cannon,199.99,90,1999.90,"Action,Adventure,Casual,Indie,Racing,RPG,Simulation,Sports,Strategy",0
68275,2499620,The Leverage Game,999.98,0,999.98,"Indie,Simulation",0
41088,2504210,The Leverage Game Business Edition,999.98,0,999.98,"Indie,Simulation",0
34463,1200520,Ascent Free-Roaming VR Experience,999.00,0,999.00,Action,0
97762,3822960,Zekertune,30.76,95,615.20,Casual,0
31935,3013840,True Love,500.00,0,500.00,"Action,Adventure,Casual,Indie",0
36737,2610220,Hidden Space Top-Down 3D,199.99,40,333.32,"Action,Adventure,Casual,Indie,Racing,RPG,Simulation,Sports,Strategy",0
84342,2623570,Hidden World War II Top-Down 3D,199.99,40,333.32,"Action,Adventure,Casual,Indie,Racing,RPG,Sports,Strategy",0
65456,3230870,Archaeology - FROZEN WALL,199.99,40,333.32,"Action,Adventure,Casual,Indie,Racing,RPG,Simulation,Sports,Strategy",0


## 3. Engineer structured features

These are compact numeric and boolean fields that can be used directly or after scaling. Skewed counts are log-transformed and clipped versions are kept for robust baselines.

In [6]:
REFERENCE_YEAR = 2026

df["platform_count"] = df[BOOL_COLS].sum(axis=1)
df["release_age_years"] = REFERENCE_YEAR - df["Release year"]
df.loc[df["release_age_years"].lt(0), "release_age_years"] = np.nan

df["review_count"] = df["Positive"].fillna(0) + df["Negative"].fillna(0)
df["positive_share"] = np.where(df["review_count"].gt(0), df["Positive"] / df["review_count"], np.nan)

df["dlc_count_clipped"] = df["DLC count"].clip(lower=0, upper=50)
df["achievements_clipped"] = df["Achievements"].clip(lower=0, upper=500)
df["required_age_clipped"] = df["Required age"].clip(lower=0, upper=21)

df["log1p_dlc_count"] = np.log1p(df["DLC count"].clip(lower=0))
df["log1p_achievements"] = np.log1p(df["Achievements"].clip(lower=0))
df["log1p_review_count"] = np.log1p(df["review_count"])
df["log1p_recommendations"] = np.log1p(df["Recommendations"].fillna(0))
df["log1p_peak_ccu"] = np.log1p(df["Peak CCU"].fillna(0))
df["log1p_avg_playtime_forever"] = np.log1p(df["Average playtime forever"].fillna(0))
df["log1p_median_playtime_forever"] = np.log1p(df["Median playtime forever"].fillna(0))

structured_cols = [
    "current_price", "estimated_list_price", "log_list_price",
    "has_active_discount", "valid_price_target", "platform_count",
    "Release year", "release_age_years", "required_age_clipped",
    "dlc_count_clipped", "achievements_clipped", "log1p_dlc_count",
    "log1p_achievements", "review_count", "positive_share",
    "log1p_review_count", "log1p_recommendations", "log1p_peak_ccu",
    "log1p_avg_playtime_forever", "log1p_median_playtime_forever",
]

df[structured_cols].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
current_price,122611.0,NaN,NaN,NaN,4.765091,12.53103,0.0,0.55,2.24,5.24,999.98
estimated_list_price,96403.0,NaN,NaN,NaN,8.720264,21.17021,0.49,2.39,4.99,9.99,3999.8
log_list_price,96062.0,NaN,NaN,NaN,1.851706,0.808687,0.398776,1.22083,1.790091,2.396986,4.615022
has_active_discount,122611,2,False,81930,NaN,NaN,NaN,NaN,NaN,NaN,NaN
valid_price_target,122611,2,True,96062,NaN,NaN,NaN,NaN,NaN,NaN,NaN
platform_count,122611.0,NaN,NaN,NaN,1.301392,0.628434,1.0,1.0,1.0,1.0,3.0
Release year,122611.0,NaN,NaN,NaN,2021.53266,3.274996,1997.0,2019.0,2022.0,2024.0,2026.0
release_age_years,122611.0,NaN,NaN,NaN,4.46734,3.274996,0.0,2.0,4.0,7.0,29.0
required_age_clipped,122611.0,NaN,NaN,NaN,0.167611,1.653591,0.0,0.0,0.0,0.0,21.0
dlc_count_clipped,122611.0,NaN,NaN,NaN,0.397599,2.353827,0.0,0.0,0.0,0.0,50.0


## 4. Split multi-label fields

`Genres`, `Tags`, and `Categories` are comma-separated multi-label fields. Splitting them into lists preserves shared labels across different combinations and prepares them for multi-hot encoding in the modeling notebook.

In [7]:
def split_items(value):
    if pd.isna(value):
        return []
    return [item.strip() for item in str(value).split(",") if item.strip()]


df["genre_list"] = df["Genres"].apply(split_items)
df["tag_list"] = df["Tags"].apply(split_items)
df["category_list"] = df["Categories"].apply(split_items)

df["genre_count"] = df["genre_list"].str.len()
df["tag_count"] = df["tag_list"].str.len()
df["category_count"] = df["category_list"].str.len()

df[["Name", "Genres", "genre_list", "Tags", "tag_list", "Categories", "category_list"]].head()

,Name,Genres,genre_list,Tags,tag_list,Categories,category_list
0,Black Dragon Mage Playtest,NaN,[],NaN,[],NaN,[]
1,Supipara - Chapter 1 Spring Has Come!,Adventure,[Adventure],"Adventure,Visual Novel,Anime,Cute","[Adventure, Visual Novel, Anime, Cute]","Single-player,Steam Trading Cards,Steam Cloud,Family Sharing","[Single-player, Steam Trading Cards, Steam Cloud, Family Sharing]"
2,Mystery Solitaire The Black Raven,Casual,[Casual],"Casual,Card Game,Solitaire,Puzzle,Hidden Object,2D,Colorful,Stylized,Logic,Mystery,Atmospheric,Family Friendly,PvE,Tutorial,Singleplayer...","[Casual, Card Game, Solitaire, Puzzle, Hidden Object, 2D, Colorful, Stylized, Logic, Mystery, Atmospheric, Family Friendly, PvE, Tutoria...","Single-player,Family Sharing","[Single-player, Family Sharing]"
3,버튜버 파라노이아 - Vtuber Paranoia,"Casual,Indie,Simulation","[Casual, Indie, Simulation]",NaN,[],"Single-player,Steam Achievements,Family Sharing","[Single-player, Steam Achievements, Family Sharing]"
4,Maze Quest VR,"Action,Early Access","[Action, Early Access]",NaN,[],"Single-player,VR Only,Steam Leaderboards,Family Sharing","[Single-player, VR Only, Steam Leaderboards, Family Sharing]"


In [8]:
def label_counts(series):
    return series.explode().loc[lambda s: s.notna() & s.ne("")].value_counts()


genre_counts = label_counts(df.loc[df["valid_price_target"], "genre_list"])
tag_counts = label_counts(df.loc[df["valid_price_target"], "tag_list"])
category_counts = label_counts(df.loc[df["valid_price_target"], "category_list"])

frequency_summary = pd.DataFrame({
    "field": ["Genres", "Tags", "Categories"],
    "unique_labels": [len(genre_counts), len(tag_counts), len(category_counts)],
    "labels_count_ge_50": [(genre_counts >= 50).sum(), (tag_counts >= 50).sum(), (category_counts >= 50).sum()],
    "labels_count_ge_100": [(genre_counts >= 100).sum(), (tag_counts >= 100).sum(), (category_counts >= 100).sum()],
    "labels_count_ge_300": [(genre_counts >= 300).sum(), (tag_counts >= 300).sum(), (category_counts >= 300).sum()],
})
frequency_summary

,field,unique_labels,labels_count_ge_50,labels_count_ge_100,labels_count_ge_300
0,Genres,33,26,23,15
1,Tags,452,418,390,325
2,Categories,56,52,49,46


In [9]:
RETAINED_TAG_MIN_COUNT = 100
RETAINED_CATEGORY_MIN_COUNT = 100

retained_genres = sorted(genre_counts.index.tolist())
retained_tags = sorted(tag_counts[tag_counts >= RETAINED_TAG_MIN_COUNT].index.tolist())
retained_categories = sorted(category_counts[category_counts >= RETAINED_CATEGORY_MIN_COUNT].index.tolist())

print(f"Retained genres: {len(retained_genres)}")
print(f"Retained tags with count >= {RETAINED_TAG_MIN_COUNT}: {len(retained_tags)}")
print(f"Retained categories with count >= {RETAINED_CATEGORY_MIN_COUNT}: {len(retained_categories)}")

pd.concat(
    [
        genre_counts.head(15).rename("genre_count"),
        tag_counts.head(15).rename("tag_count"),
        category_counts.head(15).rename("category_count"),
    ],
    axis=1,
)

Retained genres: 33
Retained tags with count >= 100: 390
Retained categories with count >= 100: 49


,genre_count,tag_count,category_count
Indie,68732.0,43417.0,NaN
Casual,42549.0,32954.0,NaN
Action,38771.0,32823.0,NaN
Adventure,38690.0,32044.0,NaN
Simulation,20692.0,16245.0,NaN
Strategy,18826.0,16007.0,NaN
RPG,17530.0,14088.0,NaN
Early Access,9241.0,NaN,NaN
Sports,4062.0,NaN,NaN
Racing,3405.0,NaN,NaN


## 5. Define modeling scopes and feature groups

The pre-release feature set is appropriate for developer pricing. The post-release set adds outcome variables for gamer-facing value assessment.

In [10]:
ID_COLS = ["AppID", "Name"]

TARGET_COLS = [
    "current_price",
    "Discount",
    "has_active_discount",
    "estimated_list_price",
    "log_list_price",
    "price_target_source",
    "valid_price_target",
]

PRE_RELEASE_NUMERIC_FEATURES = [
    "required_age_clipped",
    "platform_count",
    "Release year",
    "release_age_years",
    "dlc_count_clipped",
    "achievements_clipped",
    "log1p_dlc_count",
    "log1p_achievements",
    "genre_count",
    "tag_count",
    "category_count",
]

PRE_RELEASE_BOOLEAN_FEATURES = ["Windows", "Mac", "Linux"]
PRE_RELEASE_MULTILABEL_FEATURES = ["genre_list", "tag_list", "category_list"]

POST_RELEASE_FEATURES = [
    "review_count",
    "positive_share",
    "log1p_review_count",
    "log1p_recommendations",
    "log1p_peak_ccu",
    "log1p_avg_playtime_forever",
    "log1p_median_playtime_forever",
    "Metacritic score",
]

RAW_REFERENCE_COLS = [
    "Release date",
    "Release date parsed",
    "Estimated owners",
    "Genres",
    "Tags",
    "Categories",
    "Developers",
    "Publishers",
]

MODEL_EXPORT_COLS = (
    ID_COLS
    + TARGET_COLS
    + RAW_REFERENCE_COLS
    + PRE_RELEASE_BOOLEAN_FEATURES
    + PRE_RELEASE_NUMERIC_FEATURES
    + PRE_RELEASE_MULTILABEL_FEATURES
    + POST_RELEASE_FEATURES
)

feature_group_summary = pd.Series({
    "id_cols": len(ID_COLS),
    "target_cols": len(TARGET_COLS),
    "pre_release_numeric": len(PRE_RELEASE_NUMERIC_FEATURES),
    "pre_release_boolean": len(PRE_RELEASE_BOOLEAN_FEATURES),
    "pre_release_multilabel": len(PRE_RELEASE_MULTILABEL_FEATURES),
    "post_release": len(POST_RELEASE_FEATURES),
    "export_cols": len(MODEL_EXPORT_COLS),
})
feature_group_summary.to_frame("count")

,count
id_cols,2
target_cols,7
pre_release_numeric,11
pre_release_boolean,3
pre_release_multilabel,3
post_release,8
export_cols,42


In [11]:
scope_summary = pd.Series({
    "raw_rows": len(df),
    "paid_rows": df["current_price"].gt(0).sum(),
    "valid_price_target_rows": df["valid_price_target"].sum(),
    "observed_list_price_rows": df["price_target_source"].eq("observed").sum(),
    "reconstructed_list_price_rows": df["price_target_source"].eq("reconstructed").sum(),
    "free_rows": df["price_target_source"].eq("free").sum(),
    "invalid_discount_100_rows": df["price_target_source"].eq("invalid_discount_100").sum(),
    "estimated_list_price_gt_100_rows": df["estimated_list_price"].gt(100).sum(),
})
scope_summary.to_frame("count")

,count
raw_rows,122611
paid_rows,96405
valid_price_target_rows,96062
observed_list_price_rows,55763
reconstructed_list_price_rows,40640
free_rows,26206
invalid_discount_100_rows,2
estimated_list_price_gt_100_rows,341


## 6. Save modeling table

The saved CSV keeps list columns as Python-list strings. The next notebook can parse them with `ast.literal_eval()` before multi-hot encoding.

In [12]:
model_df = df.loc[df["valid_price_target"], MODEL_EXPORT_COLS].copy()

for col in ["genre_list", "tag_list", "category_list"]:
    model_df[col] = model_df[col].apply(repr)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
model_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(model_df):,} valid modeling rows and {model_df.shape[1]} columns to {OUTPUT_PATH}")
print("Max estimated list price:", model_df["estimated_list_price"].max())
model_df.head()

Saved 96,062 valid modeling rows and 42 columns to /Users/camomilesson/Documents/Harbour Space/ML 1/project/data/games_price_model.csv
Max estimated list price: 99.99


,AppID,Name,current_price,Discount,has_active_discount,estimated_list_price,log_list_price,price_target_source,valid_price_target,Release date,Release date parsed,Estimated owners,Genres,Tags,Categories,Developers,Publishers,Windows,Mac,Linux,required_age_clipped,platform_count,Release year,release_age_years,dlc_count_clipped,achievements_clipped,log1p_dlc_count,log1p_achievements,genre_count,tag_count,category_count,genre_list,tag_list,category_list,review_count,positive_share,log1p_review_count,log1p_recommendations,log1p_peak_ccu,log1p_avg_playtime_forever,log1p_median_playtime_forever,Metacritic score
1,496350,Supipara - Chapter 1 Spring Has Come!,5.24,65,True,14.97,2.770712,reconstructed,True,"Jul 29, 2016",2016-07-29,0 - 20000,Adventure,"Adventure,Visual Novel,Anime,Cute","Single-player,Steam Trading Cards,Steam Cloud,Family Sharing",minori,MangaGamer,True,False,False,0,1,2016,10.0,0,0,0.000000,0.000000,1,4,4,['Adventure'],"['Adventure', 'Visual Novel', 'Anime', 'Cute']","['Single-player', 'Steam Trading Cards', 'Steam Cloud', 'Family Sharing']",255,0.988235,5.545177,5.446737,0.000000,2.197225,2.197225,0
2,1034400,Mystery Solitaire The Black Raven,4.99,0,False,4.99,1.790091,observed,True,"May 6, 2019",2019-05-06,0 - 20000,Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Object,2D,Colorful,Stylized,Logic,Mystery,Atmospheric,Family Friendly,PvE,Tutorial,Singleplayer...","Single-player,Family Sharing",Somer Games,8floor,True,True,False,0,2,2019,7.0,0,0,0.000000,0.000000,1,16,2,['Casual'],"['Casual', 'Card Game', 'Solitaire', 'Puzzle', 'Hidden Object', '2D', 'Colorful', 'Stylized', 'Logic', 'Mystery', 'Atmospheric', 'Family...","['Single-player', 'Family Sharing']",24,0.875000,3.218876,0.000000,0.000000,0.000000,0.000000,0
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,8.99,0,False,8.99,2.301585,observed,True,"Oct 31, 2024",2024-10-31,0 - 20000,"Casual,Indie,Simulation",NaN,"Single-player,Steam Achievements,Family Sharing",유진게임즈,유진게임즈,True,False,False,0,1,2024,2.0,1,19,0.693147,2.995732,3,0,3,"['Casual', 'Indie', 'Simulation']",[],"['Single-player', 'Steam Achievements', 'Family Sharing']",0,NaN,0.000000,0.000000,0.693147,0.000000,0.000000,0
4,3631080,Maze Quest VR,4.99,0,False,4.99,1.790091,observed,True,"Apr 24, 2025",2025-04-24,0 - 20000,"Action,Early Access",NaN,"Single-player,VR Only,Steam Leaderboards,Family Sharing",Reality Expanded LLC,Reality Expanded LLC,True,False,False,0,1,2025,1.0,0,0,0.000000,0.000000,2,0,4,"['Action', 'Early Access']",[],"['Single-player', 'VR Only', 'Steam Leaderboards', 'Family Sharing']",0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0
5,1654170,Agony VR,13.99,0,False,13.99,2.707383,observed,True,"Apr 5, 2023",2023-04-05,0 - 20000,"Action,Adventure",NaN,"Single-player,Tracked Controller Support,VR Only,Family Sharing",Ignibit,"Ignibit,Madmind Studio",True,False,False,0,1,2023,3.0,0,0,0.000000,0.000000,2,0,4,"['Action', 'Adventure']",[],"['Single-player', 'Tracked Controller Support', 'VR Only', 'Family Sharing']",0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0


In [13]:
reloaded = pd.read_csv(OUTPUT_PATH)
for col in ["genre_list", "tag_list", "category_list"]:
    reloaded[col] = reloaded[col].apply(ast.literal_eval)

print(f"Reloaded shape: {reloaded.shape}")
print("Valid targets after reload:", reloaded["valid_price_target"].sum())
print("Invalid targets after reload:", (~reloaded["valid_price_target"]).sum())
print("Max estimated list price after reload:", reloaded["estimated_list_price"].max())
reloaded[["AppID", "Name", "current_price", "Discount", "estimated_list_price", "genre_list", "tag_list"]].head()

Reloaded shape: (96062, 42)
Valid targets after reload: 96062
Invalid targets after reload: 0
Max estimated list price after reload: 99.99


,AppID,Name,current_price,Discount,estimated_list_price,genre_list,tag_list
0,496350,Supipara - Chapter 1 Spring Has Come!,5.24,65,14.97,[Adventure],"[Adventure, Visual Novel, Anime, Cute]"
1,1034400,Mystery Solitaire The Black Raven,4.99,0,4.99,[Casual],"[Casual, Card Game, Solitaire, Puzzle, Hidden Object, 2D, Colorful, Stylized, Logic, Mystery, Atmospheric, Family Friendly, PvE, Tutoria..."
2,3292190,버튜버 파라노이아 - Vtuber Paranoia,8.99,0,8.99,"[Casual, Indie, Simulation]",[]
3,3631080,Maze Quest VR,4.99,0,4.99,"[Action, Early Access]",[]
4,1654170,Agony VR,13.99,0,13.99,"[Action, Adventure]",[]


## 7. Decisions for the modeling notebook

- Main target: `log_list_price`, derived from `estimated_list_price`, because paid-game prices are skewed and relative pricing errors are more meaningful during training.
- Price target rule: keep valid paid games, reconstruct list price from discounted rows where possible, exclude fully discounted rows, and keep the `$0.49` to `$99.99` modeling range.
- Main training scope: `valid_price_target == True`; later notebooks compare stricter market scopes such as `review_count >= 10`.
- Sensitivity scope: `price_target_source == "observed"` for non-discounted-only validation.
- First model: pre-release features only, appropriate for developer-facing launch or benchmark pricing.
- Second model: add post-release features for gamer-facing value assessment, where review and engagement signals are visible to players.
- Multi-label encoding cutoffs to start with: all genres, tags with count >= 100, categories with count >= 100.